# Day 20 — 多指标共振策略

**目标**：把前三周学的 6 个指标 + K线形态 + 背离全部串起来，制定第一个完整交易策略。

**核心概念**：单一指标胜率≈抛硬币（25%-50%），但多个独立指标同时指向同一方向时，可信度指数级上升。这就是「共振」。

In [ ]:
import pandas as pd
import numpy as np

# ===== 读取数据 =====
df = pd.read_csv('/tmp/mt.csv')
df['日期'] = pd.to_datetime(df['日期'])
df = df.set_index('日期').sort_index()

# ===== 计算所有指标 =====
# 1. 均线 MA
df['MA5'] = df['收盘'].rolling(5).mean()
df['MA20'] = df['收盘'].rolling(20).mean()
df['MA60'] = df['收盘'].rolling(60).mean()

# 2. RSI(14)
def calc_rsi(series, period=14):
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = (-delta).clip(lower=0)
    avg_gain = gain.ewm(alpha=1/period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/period, adjust=False).mean()
    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))

df['RSI'] = calc_rsi(df['收盘'], 14)

# 3. MACD
ema12 = df['收盘'].ewm(span=12, adjust=False).mean()
ema26 = df['收盘'].ewm(span=26, adjust=False).mean()
df['DIF'] = ema12 - ema26
df['DEA'] = df['DIF'].ewm(span=9, adjust=False).mean()
df['MACD柱'] = 2 * (df['DIF'] - df['DEA'])

# 4. 布林带
df['BB中轨'] = df['收盘'].rolling(20).mean()
df['BB标准差'] = df['收盘'].rolling(20).std()
df['BB上轨'] = df['BB中轨'] + 2 * df['BB标准差']
df['BB下轨'] = df['BB中轨'] - 2 * df['BB标准差']
df['%B'] = (df['收盘'] - df['BB下轨']) / (df['BB上轨'] - df['BB下轨'])

# 5. 成交量
df['量MA20'] = df['成交量'].rolling(20).mean()
df['放量'] = df['成交量'] > 1.5 * df['量MA20']

# 6. K线属性
df['实体'] = abs(df['收盘'] - df['开盘'])
df['下影'] = df[['开盘','收盘']].min(axis=1) - df['最低']
df['上影'] = df['最高'] - df[['开盘','收盘']].max(axis=1)
df['K线长度'] = df['最高'] - df['最低']
df['阳线'] = df['收盘'] > df['开盘']

print(f'数据范围: {df.index[0].date()} ~ {df.index[-1].date()}')
print(f'共 {len(df)} 个交易日')
print(f'最新收盘价: {df["收盘"].iloc[-1]:.2f}')
df.tail(3)[['收盘','MA5','MA20','RSI','DIF','%B','成交量']]

---
## 第一部分：共振信号定义

定义 6 个买入信号维度，每个维度独立判断，累加得到「共振分数」：

| 维度 | 指标 | 买入条件 | 权重 |
|------|------|----------|------|
| 趋势 | MA | MA5 > MA20（短期在长期上方） | 1分 |
| 动能 | MACD | DIF > DEA（金叉状态） | 1分 |
| 力度 | RSI | 30 < RSI < 50（脱离超卖但未过热） | 1分 |
| 位置 | BB | %B < 0.3（价格在低位） | 1分 |
| 资金 | 量 | 放量 > 1.5倍均量 | 1分 |
| 反转 | K线+背离 | 底背离 OR 看涨K线形态 | 1分 |

共振分数范围 0-6，分数越高 = 信号越强。

In [ ]:
# ===== 共振信号打分 =====
d = df.copy()

# 维度1: 趋势 — MA5在MA20上方
d['SIG_MA'] = (d['MA5'] > d['MA20']).astype(int)

# 维度2: 动能 — MACD金叉状态(DIF在DEA上方)
d['SIG_MACD'] = (d['DIF'] > d['DEA']).astype(int)

# 维度3: 力度 — RSI在30-50之间(脱离超卖但不过热)
d['SIG_RSI'] = ((d['RSI'] > 30) & (d['RSI'] < 50)).astype(int)

# 维度4: 位置 — %B < 0.3（价格在布林带低位）
d['SIG_BB'] = (d['%B'] < 0.3).astype(int)

# 维度5: 资金 — 放量（成交量 > 1.5倍20日均量）
d['SIG_VOL'] = d['放量'].astype(int)

# 维度6: 反转 — K线形态(锤子线+吞没+早晨之星)或底背离
d['锤子'] = ((d['下影'] >= 2*d['实体']) & (d['上影'] <= 0.3*d['实体']) & (d['实体'] > 0)).astype(int)
d['吞没'] = (d['阳线'] & d['阴线'].shift(1).fillna(False) & 
            (d['开盘'] < d['收盘'].shift(1)) & (d['收盘'] > d['开盘'].shift(1))).astype(int)
d['SIG_KLINE'] = ((d['锤子'] + d['吞没']) > 0).astype(int)

# 共振总分
d['共振分数'] = d['SIG_MA'] + d['SIG_MACD'] + d['SIG_RSI'] + d['SIG_BB'] + d['SIG_VOL'] + d['SIG_KLINE']

print('各维度信号统计（近120天）:')
d120 = d.iloc[-120:]
for col, name in [('SIG_MA','MA趋势'),('SIG_MACD','MACD动能'),('SIG_RSI','RSI力度'),
                   ('SIG_BB','BB位置'),('SIG_VOL','成交量'),('SIG_KLINE','K线反转')]:
    print(f'  {name}: {d120[col].sum():.0f}天有信号')

print(f'\n共振分数分布（近120天）:')
print(d120['共振分数'].value_counts().sort_index())

---
## 第二部分：共振回测

分别测试不同共振分数买入后 5日/10日/20日 的收益表现。
关键问题：分数越高，胜率是否越高？

In [ ]:
# ===== 共振策略回测 =====
def backtest_resonance(df, signal_col='共振分数', min_score=1, hold_days=5):
    """对满足共振分数要求的买入信号进行回测"""
    results = []
    # 避免未来函数：只用当天及之前的数据
    for i in range(len(df)):
        if df[signal_col].iloc[i] >= min_score:
            buy_price = df['收盘'].iloc[i]
            if i + hold_days < len(df):
                sell_price = df['收盘'].iloc[i + hold_days]
                ret = (sell_price / buy_price - 1) * 100
                results.append({
                    '日期': df.index[i],
                    '买入价': buy_price,
                    '卖出价': sell_price,
                    '收益%': ret,
                    '分数': df[signal_col].iloc[i]
                })
    return pd.DataFrame(results)

# 用最近120天数据回测
d120 = d.iloc[-120:].copy()

print('='*70)
print('茅台共振策略回测 — 不同分数阈值 × 不同持仓周期')
print('='*70)

for min_score in [1, 2, 3, 4, 5]:
    print(f'\n--- 共振分数 ≥ {min_score} ---')
    for hold in [5, 10, 20]:
        bt = backtest_resonance(d120, '共振分数', min_score, hold)
        if len(bt) > 0:
            win_rate = (bt['收益%'] > 0).sum() / len(bt) * 100
            avg_ret = bt['收益%'].mean()
            print(f'  持仓{hold}日: {len(bt)}次交易 | 胜率{win_rate:.0f}% | 均值{avg_ret:+.2f}%')
        else:
            print(f'  持仓{hold}日: 无交易信号')


---
## 第三部分：共振信号的逻辑验证

检查一个关键假设：共振分数越高 → 胜率越高。
如果分数和胜率正相关，说明共振逻辑成立。

In [ ]:
# ===== 胜率-分数关系 =====
print('胜率 vs 共振分数（持仓10日）')
print('-'*40)
for score in range(1, 7):
    bt = backtest_resonance(d120, '共振分数', score, 10)
    if len(bt) > 0:
        wr = (bt['收益%'] > 0).sum() / len(bt) * 100
        avg = bt['收益%'].mean()
        print(f'  分数≥{score}: {len(bt)}次 | 胜率{wr:.0f}% | 均值{avg:+.2f}%')
    else:
        print(f'  分数≥{score}: 无信号')


---
## 第四部分：三只股票共振对比

同一套共振系统应用在三只股票上，看谁当前信号最强。

In [ ]:
# ===== 三股共振扫描 =====
def calc_all_signals(df):
    """一次性计算所有指标和共振分数"""
    d = df.copy()
    d['MA5'] = d['收盘'].rolling(5).mean()
    d['MA20'] = d['收盘'].rolling(20).mean()
    
    ema12 = d['收盘'].ewm(span=12, adjust=False).mean()
    ema26 = d['收盘'].ewm(span=26, adjust=False).mean()
    d['DIF'] = ema12 - ema26
    d['DEA'] = d['DIF'].ewm(span=9, adjust=False).mean()
    
    d['RSI'] = calc_rsi(d['收盘'], 14)
    d['%B'] = (d['收盘'] - d['收盘'].rolling(20).mean() + 2*d['收盘'].rolling(20).std()) / (4*d['收盘'].rolling(20).std())
    d['量MA20'] = d['成交量'].rolling(20).mean()
    
    d['SIG_MA'] = (d['MA5'] > d['MA20']).astype(int)
    d['SIG_MACD'] = (d['DIF'] > d['DEA']).astype(int)
    d['SIG_RSI'] = ((d['RSI'] > 30) & (d['RSI'] < 50)).astype(int)
    d['SIG_BB'] = (d['%B'] < 0.3).astype(int)
    d['SIG_VOL'] = (d['成交量'] > 1.5 * d['量MA20']).astype(int)
    
    d['实体'] = abs(d['收盘'] - d['开盘'])
    d['下影'] = d[['开盘','收盘']].min(axis=1) - d['最低']
    d['上影'] = d['最高'] - d[['开盘','收盘']].max(axis=1)
    d['K线长度'] = d['最高'] - d['最低']
    d['阳线'] = d['收盘'] > d['开盘']
    d['锤子'] = ((d['下影']>=2*d['实体']) & (d['上影']<=0.3*d['实体']) & (d['实体']>0)).astype(int)
    d['吞没'] = (d['阳线'] & d['阴线'].shift(1).fillna(False) & 
                (d['开盘']<d['收盘'].shift(1)) & (d['收盘']>d['开盘'].shift(1))).astype(int)
    d['SIG_KLINE'] = ((d['锤子']+d['吞没'])>0).astype(int)
    
    d['共振分数'] = d['SIG_MA']+d['SIG_MACD']+d['SIG_RSI']+d['SIG_BB']+d['SIG_VOL']+d['SIG_KLINE']
    return d

print('三只股票共振信号对比（最新一天）')
print('='*70)

for name, path in [('茅台','/tmp/mt.csv'),('招商银行','/tmp/zs.csv'),('比亚迪','/tmp/byd.csv')]:
    s = pd.read_csv(path)
    s['日期'] = pd.to_datetime(s['日期'])
    s = s.set_index('日期').sort_index()
    r = calc_all_signals(s)
    last = r.iloc[-1]
    
    print(f'\n📊 {name}  收盘:{last["收盘"]:.2f}  RSI:{last["RSI"]:.1f}')
    print(f'   MA趋势:{"✅" if last["SIG_MA"] else "❌"} '
          f' MACD:{"✅" if last["SIG_MACD"] else "❌"} '
          f' RSI低位:{"✅" if last["SIG_RSI"] else "❌"} '
          f' BB低位:{"✅" if last["SIG_BB"] else "❌"} '
          f' 放量:{"✅" if last["SIG_VOL"] else "❌"} '
          f' K线反转:{"✅" if last["SIG_KLINE"] else "❌"}')
    print(f'   共振总分: {int(last["共振分数"])}/6')
    
    # 近120天高分信号统计
    rd = r.iloc[-120:]
    for score in [3,4,5,6]:
        cnt = (rd['共振分数'] >= score).sum()
        if cnt > 0:
            bt = backtest_resonance(rd, '共振分数', score, 10)
            if len(bt) > 0:
                wr = (bt['收益%'] > 0).sum()/len(bt)*100
                avg = bt['收益%'].mean()
                print(f'   分数≥{score}: {cnt}天出现 | 10日胜率{wr:.0f}% 均值{avg:+.2f}%')


---
## 第五部分：完整的交易策略

基于共振回测结果，制定第一个完整交易策略：

**买入条件（AND逻辑）：**
1. 共振分数 ≥ 3（至少三个维度同时看多）
2. MACD必须处于金叉状态（动能确认）
3. 成交量必须放量（资金确认）

**卖出条件（OR逻辑）：**
1. 持仓10个交易日自动卖出
2. 亏损超过-5%止损
3. MACD死叉（DIF下穿DEA）止盈/止损

**仓位管理：**
- 总资金的30%用于一次交易
- 其余70%留作后备（补仓或另找机会）

In [ ]:
# ===== 完整策略回测 =====
def full_strategy_backtest(df, buy_score=3, hold_days=10, stop_loss=-0.05):
    """
    完整策略：
    - 买入：共振分数≥buy_score + MACD金叉 + 放量
    - 卖出：持仓hold_days天 或 触及止损 或 MACD死叉
    """
    trades = []
    in_position = False
    buy_idx = None
    buy_price = None
    
    for i in range(len(df)):
        if not in_position:
            # 买入条件
            cond_score = df['共振分数'].iloc[i] >= buy_score
            cond_macd = df['DIF'].iloc[i] > df['DEA'].iloc[i]
            cond_vol = df['SIG_VOL'].iloc[i] == 1
            
            if cond_score and cond_macd and cond_vol:
                in_position = True
                buy_idx = i
                buy_price = df['收盘'].iloc[i]
        else:
            days_held = i - buy_idx
            current_price = df['收盘'].iloc[i]
            ret = (current_price / buy_price - 1)
            
            # 卖出条件
            sell_signal = False
            sell_reason = ''
            
            if days_held >= hold_days:
                sell_signal = True
                sell_reason = '持仓到期'
            elif ret <= stop_loss:
                sell_signal = True
                sell_reason = '止损'
            elif df['DIF'].iloc[i] < df['DEA'].iloc[i] and df['DIF'].iloc[i-1] >= df['DEA'].iloc[i-1]:
                sell_signal = True
                sell_reason = 'MACD死叉'
            
            if sell_signal:
                trades.append({
                    '买入日期': df.index[buy_idx],
                    '卖出日期': df.index[i],
                    '买入价': buy_price,
                    '卖出价': current_price,
                    '收益%': ret * 100,
                    '持仓天数': days_held,
                    '卖出原因': sell_reason,
                    '买入分数': df['共振分数'].iloc[buy_idx]
                })
                in_position = False
    
    return pd.DataFrame(trades)

# 运行完整策略
d120 = d.iloc[-120:].copy()
trades = full_strategy_backtest(d120, buy_score=3, hold_days=10, stop_loss=-0.05)

print('='*70)
print('完整交易策略回测（近120天，共振分数≥3，MACD金叉+放量）')
print('='*70)

if len(trades) > 0:
    print(f'\n总交易次数: {len(trades)}')
    win_rate = (trades['收益%'] > 0).sum() / len(trades) * 100
    avg_ret = trades['收益%'].mean()
    total_ret = (1 + trades['收益%']/100).prod() - 1
    print(f'胜率: {win_rate:.0f}%')
    print(f'平均收益: {avg_ret:+.2f}%')
    print(f'累计收益: {total_ret*100:+.2f}%')
    print(f'最大盈利: {trades["收益%"].max():+.2f}%')
    print(f'最大亏损: {trades["收益%"].min():+.2f}%')
    print(f'\n卖出原因分布:')
    print(trades['卖出原因'].value_counts().to_string())
    print(f'\n逐笔明细:')
    for _, t in trades.iterrows():
        print(f'  {t["买入日期"].date()} → {t["卖出日期"].date()} '
              f' 买入{t["买入价"]:.1f} 卖出{t["卖出价"]:.1f} '
              f' {t["收益%"]:+.2f}% [{t["卖出原因"]}]')
else:
    print('\n无符合条件的交易')

# 对比：如果不用共振，单纯看MACD金叉就买
print(f'\n对比：单纯MACD金叉买入（近120天）')
macd_only = []
for i in range(len(d120)):
    if d120['DIF'].iloc[i] > d120['DEA'].iloc[i] and d120['DIF'].iloc[i-1] <= d120['DEA'].iloc[i-1]:
        if i + 10 < len(d120):
            ret = (d120['收盘'].iloc[i+10] / d120['收盘'].iloc[i] - 1) * 100
            macd_only.append(ret)
if macd_only:
    print(f'  交易次数: {len(macd_only)}')
    print(f'  胜率: {sum(1 for r in macd_only if r>0)/len(macd_only)*100:.0f}%')
    print(f'  平均收益: {np.mean(macd_only):+.2f}%')


---
## 总结：你的第一个完整交易策略

策略名称：**「六维共振」**

买入：共振分数≥3 + MACD金叉 + 放量 → 三个条件全部满足才出手

卖出：持仓10天到期 / -5%止损 / MACD死叉 → 任一条件触发即卖

仓位：单次30%仓位，不all-in

核心理念：**宁可错过，不可做错。** 大部分时间等待，只有多指标同时指路才出手。

---

## 思考题

1. 共振分数阈值设为3是否合理？设为4会怎样？
2. 这套策略在熊市（茅台近120天）胜率如何？如果在牛市的比亚迪上呢？
3. 30%仓位意味着什么？如果连续亏3次，还剩多少本金？
4. 这个策略最大的缺陷是什么？（提示：看持仓天数分布）